<a href="https://colab.research.google.com/github/ahalsey2323405/COSC-650---Applied-LLM-Systems/blob/Week4/Week4Discussion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 4 discussion**
Schema design and Hallucinated Arguments

In [ ]:
# Gemini import
from openai import OpenAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

crossover_instructions = (
    "Use the following tool schema when a rider asks to find a driver. "
)

rider_request = (
    "I'm at Lambert Airport and I need a ride to maryville university for 3 people. I'd prefer an SUV at 1:30pm."
)

loose_tools = [
    {
        "type": "function",
        "function": {
            "name": "find_driver_options",
            "description": "Find local driver options for a rider.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pickup_location": {
                        "type": "string",
                        "description": "Where the rider wants to be picked up.",
                    },
                    "destination": {
                        "type": "string",
                        "description": "Where the rider wants to go.",
                    },
                    "vehicle_type": {
                        "type": "string",
                        "description": "The type of vehicle requested.",
                    },
                    "passengers": {
                        "type": "string",
                        "description": "Number of passengers.",
                    },
                    "preferred_time": {
                        "type": "string",
                        "description": "When the rider wants the ride.",
                    },
                },
            },
        },
    }
]

resp = client.chat.completions.create(
    model="gemini-3.6-flash",
    messages=[
        {"role": "system", "content": crossover_instructions},
        {"role": "user", "content": rider_request},
    ],
    tools=loose_tools,
    tool_choice={
        "type": "function",
        "function": {"name": "find_driver_options"},
    },
)

print(resp.choices[0].message.tool_calls)



[ChatCompletionMessageFunctionToolCall(id='call_89289', function=Function(arguments='{"preferred_time":"1:30pm","vehicle_type":"SUV","pickup_location":"Lambert Airport","destination":"maryville university","passengers":"3"}', name='find_driver_options'), type='function', extra_content={'google': {'thought_signature': 'Er0FCroFARFNMg9UEakie6GdimlcYr2G8Jhd4njOG3yQqh/XjQTTwXnKqdwNV/yMJKQoTfm81ZcPViK8p53Jn1EPTurf0HNsEY4ORq6X2yftPNxrUbA898Jo1V4UMy68I3sBEZBO7HFN2qO2k9RtlRK+O+GPrE8W+ZJTesHtiQDOcU3nTh07ypin1eIe/Xlu1R2I0yvEmU11fxipAnf5+mEeotR/y/f1keYGtX3CafCZD0eACwHqUYihaIPJi/XvYsnLCMjLauS/d3CY5c/F1lsyFqKnc4XVKqzS/GjyAH2KCRVsXd3w5HCgwh23NShxSfLCqNsYQCEZf5+QvCpOjdJyCLtOY2eiMCysbFWQf9ykPZmkc0VDhaV1I6kt25iSOMmVeCekyRdpNgJqQhY6NkcngJiEISbvKz0ZWH7KpHYdyZ8hlLUi5S/LNP9HQ0ssWSee3bwO7a4eDEhhWb1EmZecaLr+aGLfVKp62mHrpSrP9OQrVc/x5qs0I8JpJa3axIOZYvYO2h3hMY1j3kcLgU3yLt4HFu/tB8sr+QTP23TokC93vjNF+kZ+rVlyFop4MONeL+R468n+01v/7BDcSGkYqOxYJdMqRVSUBDIs6hw4+nqxRLwwBeX1yS9q3Ym0aZE4pSeNyWSTJWMaVvM9FGvI/1OXUgCFWbS4Zs5R

In [ ]:
#Tightened tools revision

tight_tools = [
    {
        "type": "function",
        "function": {
            "name": "find_driver_options",
            "description": (
                "Find available driver options matching the rider's request."
            ),
            "parameters": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "pickup_location": {
                        "type": "string",
                        "description": "Rider's requested pickup location."
                    },
                    "destination": {
                        "type": "string",
                        "description": "Rider's requested destination."
                    },
                    "vehicle_type": {
                        "type": "string",
                        "enum": ["sedan", "suv", "minivan", "truck"],
                        "description": "Requested vehicle category."
                    },
                    "passenger_count": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 6,
                        "description": "Number of passengers."
                    },
                    "preferred_time": {
                        "type": "string",
                        "description": (
                            "Requested ride time, preserving vague expressions, such as 'this evening' when no exact time is given."
                        )
                    }
                },
                "required": [
                    "pickup_location",
                    "destination",
                    "vehicle_type",
                    "passenger_count",
                    "preferred_time"
                ]
            }
        }
    }
]


resp = client.chat.completions.create(
    model="gemini-3.6-flash",
    messages=[
        {"role": "system", "content": crossover_instructions},
        {"role": "user", "content": rider_request},
    ],
    tools=tight_tools,
    tool_choice={
        "type": "function",
        "function": {"name": "find_driver_options"},
    },
)

print(resp.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_259990', function=Function(arguments='{"vehicle_type":"suv","preferred_time":"1:30pm","passenger_count":3,"destination":"maryville university","pickup_location":"Lambert Airport"}', name='find_driver_options'), type='function', extra_content={'google': {'thought_signature': 'Ev8FCvwFARFNMg/IUAowF4tIieuUUKcfHunkfQCeCqSME7Z0NP9REo3WLm/VustZACQASe93qf/3kqWB/kXArR3SJGjRT8Z8xK20Ks4hVDKyOfE5DVXz1lDmhkjJJY3YTRu4CnCtOzXII2XjFz2cd++NGZLsU3ThilTLqAcNBQwUTngtzwdv/tMgd5YSg0iqHJk0oB7F3XS9fD9n6QwApZ1j8TiW1yh7AY94V7WQytx3uyUzcPzxf9O2xOOocAsnoolIEjYkRVjWU/PeCjg2QKDeGY1nortRSLvBwccCgW2mnRBrtPrrmiUaBbUvdGZziul20eRf3ChOl8u3hTYHNkaH7RjhyOryMj5BDKjjKFkffNcWiuACR3YX0tBL09HDHO5Nq+SDAJQ+BSir1GujhKH85TPTkyVUvLfWibMHHnqGFaMTK1wKTVaNpgxP3tTEzftoeXGWZsIpti/0IKVoF46z/dpKO5x+WKZzc8UEN93NzRN5NVFRxOmWttHIIOCkePYq9c6NROrSu/mch2RcRjKbNhm3O4XQQt4Yz5k9dlzdmyfunue4DLK+he4ZgED/d0Wh4+fEGfZ4G/Z0XKY1k+QxeiCUyH+MSHKO57gzKS9GYtoGq+/w1PDWKmqqxqUmMFZsPRlIjaO3e745Xpucc1qmiSCBsoV1VEfF